In [183]:
%matplotlib inline

In [184]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt


# Bulgarian Electricity Prices: A Statistical Analysis (2016–2025)

## 1. Introduction

Electricity is the most time-sensitive commodity in a modern economy. Unlike grain or oil, it cannot be stored cheaply at scale; supply must equal demand at every instant, and the price that emerges from this constraint reflects the marginal cost of producing the last megawatt-hour needed to keep the grid balanced. When the weather changes, both sides of that balance shift — heating and cooling demand respond to temperature, wind generation responds to wind, hydro reservoirs respond to precipitation — and the wholesale price moves in response. The relationship between weather and wholesale electricity price is one of the most direct, observable links between the natural environment and a market.

This project examines that relationship in the Bulgarian context, using daily data from October 2016 through December 2025.
### 1.1 The Bulgarian energy market context

Bulgaria's electricity market is small by European standards, but it has all the structural features of a modern liberalised market: a wholesale exchange (IBEX), participation in the European day-ahead coupling mechanism, a generation mix that is geographically and technologically varied (nuclear, lignite, hydro, wind, solar), and exposure to international gas and carbon prices. It is also a market in transition. Over the analysis window, Bulgaria has lived through the launch of its day-ahead exchange, the COVID demand shock, the 2021–2022 European energy crisis, and the post-crisis adjustment.

Two features of the Bulgarian market shape this analysis specifically. The wholesale day-ahead price — the variable studied here — is set on IBEX, which began clearing trades on 1 October 2016. This launch date sets the lower bound of the analytical window: there is no Bulgarian day-ahead price data before October 2016 because no liquid day-ahead market existed before then. Separately, household electricity prices in Bulgaria are largely regulated by KEVR (the Energy and Water Regulatory Commission) rather than set by the market. This means the weather-to-price relationship visible in the data describes how the wholesale market responds to weather, not how household bills respond. Section 2.1 develops this distinction in detail.

### 1.2 Literature Review

The relationship between weather variables and electricity prices is well-established in energy economics research. Boogen et al. (2024) estimate the effect of temperature, wind speed, solar radiation, and precipitation on wholesale electricity prices across six European countries, finding that weather impacts prices in a nonlinear manner and identifying thresholds at which weather effects amplify significantly. Haluška et al. (2025) analyze meteorological variables including temperature, wind speed, and humidity on electricity prices across Central European countries using ENTSO-E data from 2019 to 2024, finding that results are country-specific and asymmetric. Trebbien et al. (2024) provide a comprehensive analysis of patterns and correlations in European day-ahead electricity prices between 2019 and 2023, noting the period was characteristically abnormal due to the energy crisis and that weather is a common driving factor across bidding zones.

No study to our knowledge focuses specifically on Bulgaria over a decade-long period that includes the 2021–2022 energy crisis. This analysis aims to fill that gap.

### 1.3 Research Questions
The analysis is built around three explicitly numbered questions:

1. **How strongly do weather variables correlate with real (inflation-adjusted) Bulgarian day-ahead electricity prices?**
2. **Which weather variable has the strongest independent relationship with price?**
3. **Did the 2021–2022 energy crisis fundamentally alter the weather–price relationship, and if so, how?**

The first question establishes the basic statistical relationship. The second sharpens it by separating out variables that are correlated *because* they correlate with each other (temperature and cloud cover, for example) from variables that contribute independently. The third treats the energy crisis as a natural experiment: a period when external shocks to gas prices and supply disrupted the normal relationship between domestic conditions and domestic prices, and asks whether the disruption persisted, reversed, or simply faded.

### 1.4 Scope and Limitations

This is a correlational study, not a causal one. Weather and price are observed together in the data; nothing here establishes that weather *causes* price changes, only that the two move together (or fail to) under the conditions present in the dataset. The mechanisms by which weather affects electricity prices — heating and cooling demand, renewable generation, fuel substitution — are well understood physically and economically, and the correlations measured here are consistent with those mechanisms. But the analysis itself cannot rule out confounding factors that move with weather (seasonal demand patterns, fuel price cycles, holiday effects), and it does not attempt to.

The wholesale–retail distinction noted above also bounds the analysis. The findings describe how the *Bulgarian wholesale market* prices weather risk, not how Bulgarian households experience weather-driven price changes at the meter. The retail tariff, by design, insulates households from short-term wholesale movements.

Finally, the analysis is observational. The data records what happened; it does not say what would have happened under different conditions. Statements about the energy crisis "altering the relationship" describe what is visible in the before-and-after data, not what would have been true in a counterfactual world without the crisis.

### 1.5 Assumptions

This analysis rests on a small number of explicit assumptions. Each is stated here so that any reader can evaluate whether it holds, and so that any limitation in the conclusions can be traced back to the assumption that produced it.

- **Wholesale market prices reflect supply-and-demand response to weather.** The day-ahead clearing price is treated as the variable on which weather conditions can plausibly act. Regulated retail tariffs and long-term bilateral contracts are not part of this analysis.
- **The five chosen cities are an adequate proxy for national weather.** Sofia, Plovdiv, Varna, Burgas, and Ruse are assumed to span Bulgaria's main population centres and climate zones well enough to construct a national index. Smaller cities and rural areas are not directly represented.
- **Population is an acceptable proxy for electricity demand.** Cities with more people are assumed to contribute more to national electricity demand. The true weighting — by actual electricity consumption — is not publicly available at the regional level.
- **Population weights are stable enough to fix at a single reference year.** Bulgarian city populations did shift over the analysis window, but the shifts are assumed to be small relative to the underlying proxy uncertainty. Static weights anchored to 2020 are used throughout.
- **HICP captures the inflation relevant to euro-denominated electricity prices.** The Bulgarian HICP is assumed to be the appropriate deflator. A producer price index or energy-specific deflator might give different real-price values; the choice of consumer-side HICP is consistent with how end users experience prices.
- **ERA5 reanalysis is treated as ground truth for historical weather.** ERA5 is a model output rather than a direct measurement, but it is assumed to represent past weather faithfully enough for the daily, country-aggregated resolution used here.
- **Daily resolution is fine enough to capture the weather–price relationship.** Intraday dynamics — the morning–evening peak structure, hourly weather extremes — are aggregated away. The research questions are about multi-year patterns, and daily resolution is assumed sufficient for those.
- **The October 2016 – December 2025 window contains enough variation to support the analysis.** The window deliberately includes the COVID shock and the 2021–2022 energy crisis. These are treated as natural experiments rather than excluded as outliers, on the assumption that what the analysis can learn from them is more valuable than the cleanliness lost by including them.
- **Statistical association is the claim being made — not causation.** Every result in this analysis is correlational. Weather and price are observed together; nothing here establishes that weather *causes* price changes, only that the two move together (or fail to) in the data.

## 2. Data Sources

This analysis draws on three independent data sources, combined to form a single daily time series spanning October 2016 through December 2025. The three sources answer different questions that the others cannot: *what did electricity cost on the wholesale market?*, *what was the weather?*, and *what was a euro actually worth?* Eurostat additionally provides population data for the five cities used to construct the national weather index, described alongside the HICP. No single source could answer all of these questions, and substituting any one of them with a derived or modeled equivalent would compromise the analysis.

### 2.1 Source 1 — Ember: Day-Ahead Electricity Prices

Ember is an independent climate and energy think tank that publishes a curated dataset of European wholesale day-ahead electricity prices, sourced originally from ENTSO-E's Transparency Platform. The dataset provides daily clearing prices in nominal euros per megawatt-hour (€/MWh) for Bulgaria.

The data series for Bulgaria begins on 1 October 2016 — the date IBEX, the Bulgarian Independent Energy Exchange, launched and began clearing day-ahead trades. There is no Bulgarian day-ahead price data before this date because no liquid wholesale day-ahead market existed in Bulgaria before then. This sets the lower bound of the analysis window.

The day-ahead market is the central price-discovery mechanism for *wholesale* electricity in Bulgaria. Each day, generators and large buyers submit bids for the following day's hourly delivery, and IBEX clears them into a single price per hour. Ember aggregates these hourly clearing prices into a daily mean, which is the variable used throughout this notebook. Daily resolution is appropriate here because the weather data will also be aggregated to daily resolution, and because the research questions concern multi-year patterns rather than intraday dynamics.

It is worth being explicit about what this price represents and what it does not. The day-ahead price is the wholesale market clearing price — what generators receive for selling electricity to large buyers (utilities, industrial consumers, traders) one day in advance. It reflects the marginal cost of the last unit of electricity needed to meet expected demand, typically a gas-fired plant during peak hours. It does not reflect what Bulgarian households pay on their electricity bills. Household prices in Bulgaria are largely regulated by KEVR (the Energy and Water Regulatory Commission), updated only a few times per year, and include network charges, taxes, and supplier margins on top of the underlying energy cost. Long-term bilateral contracts negotiated outside the exchange are also not reflected in this dataset.

This distinction matters for interpreting the results. The analysis will reveal how the *wholesale market* prices weather risk — how generators and buyers respond to changing supply and demand conditions driven by temperature, wind, precipitation, and so on. It will not reveal how weather affects what consumers pay at the meter, because the regulated retail tariff insulates Bulgarian households from short-term wholesale fluctuations. The wholesale market is the appropriate place to study weather-price relationships precisely because it is the part of the system that responds to weather; the retail tariff, by design, smooths these signals out.

- **Provider:** Ember
- **License:** CC-BY-4.0 (free reuse with attribution)
- **URL:** [ember-energy.org/data/european-wholesale-electricity-price-data](https://ember-energy.org/data/european-wholesale-electricity-price-data)
- **Original source:** ENTSO-E Transparency Platform (Bulgarian market: IBEX)
- **Variable used:** Daily mean day-ahead clearing price, nominal €/MWh
- **Coverage:** 2016-10-01 through 2025-12-31
### 2.2 Source 2 — Open-Meteo: Historical Weather

Open-Meteo provides a free historical weather API that serves reanalysis data from the ERA5 dataset, produced by the European Centre for Medium-Range Weather Forecasts (ECMWF) under the Copernicus Climate Change Service. Reanalysis data is not raw observation — it is the output of a physics-based atmospheric model that ingests observations from weather stations, satellites, and balloons, and produces a globally consistent gridded estimate of past weather. This is the standard approach for historical climate research and is preferable to relying on a single weather station, which may have gaps, instrument changes, or local microclimate effects unrepresentative of the broader region.

Five variables are pulled at hourly resolution and aggregated to daily values: temperature (mean), precipitation (sum), wind speed (mean), cloud cover (mean), and snowfall (sum). The aggregation method is chosen to match the physical meaning of each variable — precipitation and snowfall accumulate, while temperature, wind, and cloud cover are averaged.

Although the Open-Meteo source has weather data going back decades, it is filtered to October 2016 onward to align with the price series. Weather observations from periods when no price data exists cannot enter the analysis, since the analysis is built around the weather–price relationship.

#### Why five cities, not one

Bulgarian electricity demand is not uniform across the country. A national-scale weather variable is needed to correlate with a national-scale price. Using only Sofia would bias the analysis toward conditions in the capital and miss the contributions of coastal cities (where summer cooling demand peaks) and the Danube plain (which has its own climate regime). Weather is therefore pulled for five Bulgarian cities — Sofia, Plovdiv, Varna, Burgas, and Ruse — chosen to span the country's main population centres and climate zones. The cities are then combined into a single national weather index. The construction of that index, including the weighting scheme and its limitations, is documented in Section 3.

- **Provider:** Open-Meteo (Historical Weather API)
- **License:** CC-BY-4.0
- **URL:** [open-meteo.com/en/docs/historical-weather-api](https://open-meteo.com/en/docs/historical-weather-api)
- **Underlying dataset:** ERA5 reanalysis (ECMWF / Copernicus Climate Change Service)
- **Variables used:** Temperature, precipitation, wind speed, cloud cover, snowfall
- **Resolution:** Hourly, aggregated to daily
- **Cities:** Sofia, Plovdiv, Varna, Burgas, Ruse
- **Coverage:** 2016-10-01 through 2025-12-31
### 2.3 Source 3 — Eurostat: Inflation and Population

Eurostat is the statistical office of the European Union. It publishes harmonised statistics across member states, drawing on data collected by each country's National Statistical Institute — for Bulgaria, the National Statistical Institute (NSI Bulgaria). Two pieces of data from Eurostat are used in this analysis. They serve different purposes but share a single publisher, a single license, and a single citation chain, and so are described together here. The HICP is loaded as a CSV; the population values are small enough to be entered inline in the notebook (five numbers, held fixed across the analysis window), with the Eurostat source cited at the point of use.
#### 2.3.1 HICP — Harmonised Index of Consumer Prices

The HICP is the official inflation measure published by Eurostat for all EU member states. National HICP data are collected by NSI Bulgaria and harmonised by Eurostat under a common methodology that makes the indices comparable across countries. The harmonised methodology is the appropriate index basis here, because Bulgarian wholesale electricity is sold on euro-denominated markets, and the HICP reflects the same currency basis as the price data.

The role of this dataset in the analysis is structural, not exploratory. Nominal prices in 2025 cannot be compared directly to nominal prices in 2016, because a euro in 2025 buys substantially less than a euro in 2016. Failing to deflate the price series would mean that any apparent long-term price trend would conflate two distinct phenomena: real changes in the cost of electricity (the thing we want to study) and the general erosion of the euro's purchasing power (which has nothing to do with weather). Section 3 documents the deflation procedure and shows the difference it makes to the price series.

The HICP is published monthly. Daily prices are deflated using the HICP value for the month in which each price observation falls — this is the standard approach and is appropriate given that the index does not change at daily resolution.

- **Variable used:** Monthly HICP index for Bulgaria, base year 2015 = 100
- **URL:** [ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX](https://ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX)
- **Coverage:** 2016-10 through 2025-12
#### 2.3.2 City Population

Population data for the five cities used in the weather index — Sofia, Plovdiv, Varna, Burgas, and Ruse — comes from Eurostat's Urban Audit (dataset `urb_cpop1`). It is used in a supporting role only: to assign weights when combining the five city-level weather series into a single national weather index. Population is used as a proxy for electricity demand, on the reasoning that cities with more people contribute more to national demand. This is a defensible-not-correct choice, and its limitations are discussed in Section 3.

A single reference year (2020) is used to compute the weights, which are then held fixed across the entire analysis period. Bulgarian city populations did shift modestly across the window — Sofia gained residents while smaller cities lost them — but these shifts are small relative to the proxy nature of population-as-demand-weight. Allowing the weights to vary year by year would couple the weather index to demographic drift in a way that complicates the interpretation of the index across time. Holding the weights fixed prioritises a stable, interpretable index over marginal demographic precision. This decision is documented explicitly in Section 3.

The five population values were pulled once from the Eurostat API and saved as a small CSV in the project's `data/` directory, alongside the other source files. Subsequent notebook runs read from this CSV directly. The reproducibility chain is intact — anyone can regenerate the CSV by re-running the original Eurostat fetch (preserved in the notebook's git history), and the values themselves are referenced by both Eurostat dataset code and city code in the source comment.

- **Variable used:** Population for Sofia, Plovdiv, Varna, Burgas, Ruse, reference year 2020
- **Source dataset:** [ec.europa.eu/eurostat/databrowser/view/URB_CPOP1](https://ec.europa.eu/eurostat/databrowser/view/URB_CPOP1)
- **Local cache:** `data/city_population_2020.csv`

### 2.4 Why these sources are independent and appropriate

The project requires at least two independent data sources; this analysis uses three. Each is produced by a different organisation, using different methods, on different phenomena. Ember curates market data from ENTSO-E. Open-Meteo serves ERA5 reanalysis from ECMWF. Eurostat publishes harmonised statistics compiled by national statistical institutes.

Independence matters here because the analysis hinges on relationships *between* sources. If the weather data and the price data came from a common underlying model, any correlation found between them could be an artifact of the shared source rather than a genuine market signal. The two sources whose relationship is the central object of study — Ember (prices) and Open-Meteo (weather) — are produced by completely separate organisations using completely separate methods, with no shared upstream pipeline.

All three sources are licensed for free reuse with attribution, which is documented in the References section. The full citation chain — Ember → ENTSO-E → IBEX, Open-Meteo → ERA5 (ECMWF / Copernicus), Eurostat → NSI Bulgaria — is preserved so that any reader can trace back to the original observations.

## 3. Data Loading & Preprocessing

This section documents how the raw data — Ember prices, Open-Meteo weather for five cities, Eurostat HICP, and Eurostat city population — are loaded, cleaned, and combined into a single analytical dataset. In total, eight CSV files are loaded from disk (one price file, five weather files, one HICP file, one population file) drawn from three providers.
### 3.1 Loading the raw files

The seven CSV files are stored in the project's `data/` directory and loaded with `pandas`. The first step on each file is the same: inspect the shape, the column types, the date range, and the count of missing values per column. This is not a formality — each dataset has its own quirks, and the inspection step is what surfaces them before they cause problems downstream.

The Ember price file is a *multi-country* dataset — it contains day-ahead clearing prices for every European country in a single table, with a country column distinguishing them. Since this analysis concerns Bulgaria only, the file is filtered to Bulgarian rows immediately on load, before any inspection or transformation. Filtering at the loading stage rather than later means every subsequent step — missingness checks, deflation, merging — operates on the variable actually being studied, and the count of NaNs in Section 3.2 reflects gaps in the Bulgarian series specifically rather than gaps that could belong to any of the 30+ countries in the raw file.

After filtering, the Ember data contains daily day-ahead clearing prices for Bulgaria in nominal €/MWh, indexed by date. The five Open-Meteo files (one per city) contain daily weather observations — see Section 3.3 for how the hourly source data was aggregated. The Eurostat HICP file contains the monthly Bulgarian inflation index, with the value held constant across each month. The Eurostat population file contains 2020 population for the five cities used in the weather index — these values are loaded here alongside the other sources, and the weights they produce are constructed in Section 3.4.

The analytical window runs from 1 October 2016 through 31 December 2025. The starting boundary is determined by the Bulgarian day-ahead price series itself: IBEX, the Bulgarian Independent Energy Exchange, launched on 1 October 2016, and there is no Bulgarian wholesale day-ahead price data before that date. Weather, HICP, and population data are all available from earlier years, but they cannot enter the analysis without a corresponding price observation, so the merged dataset is bounded by the price series. No data is discarded for convenience — the start date reflects the actual existence of the market being studied. The choice to begin in October 2016 (a partial year) rather than rounding up to January 2017 is deliberate: the three months at the start represent real observations of a real market, and excluding them would trade transparency for cosmetic round numbers.

Date columns are parsed into proper `datetime` types at load time rather than left as strings. This is the single most important preprocessing step for a time-series project: every subsequent merge, filter, and groupby depends on dates being comparable as dates, not as strings that happen to look like dates.

In [185]:
electricity_price = pd.read_csv("data/european_wholesale_electricity_price_data_daily.csv")
electricity_price['Date'] = pd.to_datetime(electricity_price['Date'])
electricity_price = electricity_price[electricity_price["Country"] == "Bulgaria"]
electricity_price = electricity_price[electricity_price['Date'] < '2026-01-01']
print(f"Prices: {electricity_price.shape}, {electricity_price['Date'].min()} to {electricity_price['Date'].max()}")

Prices: (3379, 4), 2016-10-01 00:00:00 to 2025-12-31 00:00:00


In [186]:
cities = ['sofia', 'plovdiv', 'varna', 'burgas', 'ruse']
weather = {c: pd.read_csv(f'data/weather_{c}.csv', parse_dates=['date']) for c in cities}
for c in cities:
    weather[c] = weather[c][weather[c]['date'].between('2015-01-01', '2025-12-31')]
    print(f"Weather ({c}): {weather[f'{c}'].shape}")

Weather (sofia): (4018, 6)
Weather (plovdiv): (4018, 6)
Weather (varna): (4018, 6)
Weather (burgas): (4018, 6)
Weather (ruse): (4018, 6)


In [187]:
hicp = pd.read_csv('data/prc_hicp_midx_page_linear.csv', parse_dates=['TIME_PERIOD'])
hicp = hicp[hicp['TIME_PERIOD'].between('2015-01-01', '2025-12-31')]
print(f"HICP: {hicp.shape}")

HICP: (132, 10)


In [188]:
populations = pd.read_csv("data/city_population_2020.csv")
populations = populations.set_index('city')

### 3.2 Per-source missingness check

Before any transformation, each loaded dataset is checked for missing values on its own terms. This is the first of the two missingness moments described above: missingness that exists in the raw source itself, independent of any downstream processing. The point of doing this *before* any further work is that subsequent steps — aggregation, weighting, deflation, merging — will obscure the origin of any NaN they encounter, and it becomes impossible to diagnose missingness once it has been mixed into a derived column.

Each source is checked separately, and the response to any missingness depends on what the missingness *means* in that source.

- **Ember prices.** Occasional days are missing from the Bulgarian day-ahead price series. These correspond to gaps in ENTSO-E reporting rather than to days on which the market failed to clear. Imputation is rejected for this source: substituting a fictitious price into the variable being studied would compromise the analysis. Missing price days are flagged here and dropped from the merged dataset later in Section 3.6.
- **Open-Meteo weather.** Reanalysis data is gridded model output with global coverage; per-source missingness is not expected, and inspection confirms none is present in any of the five city files. No action is required.
- **Eurostat HICP.** The Bulgarian HICP series is complete across the analysis window. No action is required.
- **Eurostat population.** Annual population values for the five cities are present. No action is required.

The result of this step is a clear inventory: *if a NaN appears later in the pipeline, the price series is the only place it could have originated, and the count of price-side gaps is known up front*. This will matter when interpreting the post-merge check in Section 3.6.

In [189]:
electricity_price.isnull().sum()

Country             0
ISO3 Code           0
Date                0
Price (EUR/MWhe)    0
dtype: int64

In [190]:
for c in cities: 
    print(weather[c].isnull().sum()) 

date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64


Drop empty Eurostat metadata columns (OBS_FLAG, CONF_STATUS).
These are SDMX export defaults; no observations in this window are flagged.

In [191]:
hicp.drop("OBS_FLAG" , axis='columns', inplace=True)
hicp.drop("CONF_STATUS", axis='columns', inplace=True)

In [192]:
hicp.isnull().sum()

DATAFLOW       0
LAST UPDATE    0
freq           0
unit           0
coicop         0
geo            0
TIME_PERIOD    0
OBS_VALUE      0
dtype: int64

### 3.3 Daily weather data

The Open-Meteo source provides weather observations at hourly resolution. For this project, the hourly data was aggregated to daily resolution outside the notebook before loading, and the daily CSVs are what enters the analysis here. This was a practical choice: the hourly files are large, the analysis operates entirely at daily resolution, and re-aggregating on every notebook run would add substantial loading time without changing any result.

The aggregation method was chosen variable by variable to match the physical meaning of each one:

- **Temperature** is averaged across the 24 hourly values for the day. The daily mean is the most common choice in climate and energy studies.
- **Precipitation** is summed. A daily precipitation total of 10 mm is meaningful; a daily precipitation average is not.
- **Snowfall** is summed, for the same reason as precipitation.
- **Wind speed** is averaged. Daily peak gust would be physically meaningful too, but mean wind speed is the standard variable for energy-system studies because wind generation depends on sustained wind, not on isolated gusts.
- **Cloud cover** is averaged. Cloud cover is reported as a percentage at each hour, and the daily mean represents the share of the day that was cloudy.

This step was performed independently for each of the five cities, producing five daily CSVs that are loaded and merged in the notebook itself.

### 3.4 Constructing the national weather index

The five city-level daily weather series are now combined into a single national weather series. The combination uses a weighted mean: each city's weather contributes to the national value in proportion to its population.

Population data was loaded in Section 3.1 alongside the other source files. Here, those values are converted into weights — each city's weight is its population divided by the total population across the five cities — and the weights are then applied to the daily weather series. The formula: for any weather variable on any day, the national value is the sum across the five cities of `weight × city_value`, where the weights sum to one.

In [193]:
populations['weight'] = populations['population'] / populations['population'].sum()
assert populations.weight.sum() - 1 == 0

temp_national = (
    weather['sofia']['temperature_c']   * populations.loc['Sofia',   'weight'] +
    weather['plovdiv']['temperature_c'] * populations.loc['Plovdiv', 'weight'] +
    weather['varna']['temperature_c']   * populations.loc['Varna',   'weight'] +
    weather['burgas']['temperature_c']  * populations.loc['Burgas',  'weight'] +
    weather['ruse']['temperature_c']    * populations.loc['Ruse',    'weight']
)
precip_national = (
    weather['sofia']['precipitation_mm']   * populations.loc['Sofia',   'weight'] +
    weather['plovdiv']['precipitation_mm'] * populations.loc['Plovdiv', 'weight'] +
    weather['varna']['precipitation_mm']   * populations.loc['Varna',   'weight'] +
    weather['burgas']['precipitation_mm']  * populations.loc['Burgas',  'weight'] +
    weather['ruse']['precipitation_mm']    * populations.loc['Ruse',    'weight']
)

wind_national = (
    weather['sofia']['windspeed_kmh']   * populations.loc['Sofia',   'weight'] +
    weather['plovdiv']['windspeed_kmh'] * populations.loc['Plovdiv', 'weight'] +
    weather['varna']['windspeed_kmh']   * populations.loc['Varna',   'weight'] +
    weather['burgas']['windspeed_kmh']  * populations.loc['Burgas',  'weight'] +
    weather['ruse']['windspeed_kmh']    * populations.loc['Ruse',    'weight']
)
cloud_national = (
    weather['sofia']['cloudcover_pct']   * populations.loc['Sofia',   'weight'] +
    weather['plovdiv']['cloudcover_pct'] * populations.loc['Plovdiv', 'weight'] +
    weather['varna']['cloudcover_pct']   * populations.loc['Varna',   'weight'] +
    weather['burgas']['cloudcover_pct']  * populations.loc['Burgas',  'weight'] +
    weather['ruse']['cloudcover_pct']    * populations.loc['Ruse',    'weight']
)
snow_national = (
    weather['sofia']['snowfall_cm']   * populations.loc['Sofia',   'weight'] +
    weather['plovdiv']['snowfall_cm'] * populations.loc['Plovdiv', 'weight'] +
    weather['varna']['snowfall_cm']   * populations.loc['Varna',   'weight'] +
    weather['burgas']['snowfall_cm']  * populations.loc['Burgas',  'weight'] +
    weather['ruse']['snowfall_cm']    * populations.loc['Ruse',    'weight']
)
weather_national = pd.DataFrame({
    'Date': weather['sofia']['date'],
    'temperature':   temp_national,
    'precipitation': precip_national,
    'wind_speed':    wind_national,
    'cloud_cover':   cloud_national,
    'snowfall':      snow_national,
})
weather_national

,Date,temperature,precipitation,wind_speed,cloud_cover,snowfall
0,2015-01-01,-11.117692,0.000000,10.245885,32.440075,0.012446
1,2015-01-02,-6.369779,0.024890,8.441764,37.243604,0.032149
2,2015-01-03,-1.335975,0.109494,10.155669,61.914304,0.153292
3,2015-01-04,0.465656,1.474759,10.942830,82.567123,0.210737
4,2015-01-05,-2.537559,0.164242,12.527105,35.094398,0.229938
...,...,...,...,...,...,...
4013,2025-12-27,1.515136,0.000000,16.067292,67.792444,0.000000
4014,2025-12-28,2.160068,0.121775,19.473101,43.649332,0.000000
4015,2025-12-29,1.605713,0.000000,16.154065,18.378453,0.000000
4016,2025-12-30,1.068996,0.000000,13.083620,16.341491,0.000000


In [194]:
weather['sofia'] 

,date,temperature_c,precipitation_mm,windspeed_kmh,cloudcover_pct,snowfall_cm
0,2015-01-01,-14.245833,0.0,5.891667,33.125000,0.00
1,2015-01-02,-10.158333,0.0,5.487500,20.791667,0.00
2,2015-01-03,-3.783333,0.2,8.541667,69.041667,0.28
3,2015-01-04,-1.129167,0.7,10.308333,87.791667,0.00
4,2015-01-05,-4.425000,0.3,10.704167,51.500000,0.42
...,...,...,...,...,...,...
4013,2025-12-27,0.587500,0.0,15.900000,85.083333,0.00
4014,2025-12-28,1.087500,0.0,19.258333,52.541667,0.00
4015,2025-12-29,0.616667,0.0,15.141667,29.541667,0.00
4016,2025-12-30,-0.483333,0.0,12.245833,17.708333,0.00


### 3.5 Deflating prices to real terms

The Ember price series is in nominal €/MWh — the actual euro figure that changed hands on the day. Across a multi-year window, nominal prices conflate two things: real changes in the cost of electricity, and the general erosion of the euro's purchasing power. To isolate the first, prices are deflated using the standard formula:

$$P^{\text{real}}_t = P^{\text{nominal}}_t \times \frac{\text{HICP}_{\text{base}}}{\text{HICP}_t}$$

The base year is 2015, Eurostat's standard reference period — independent of the analysis window. Each daily price is deflated using the HICP value for its calendar month, since HICP is published monthly. The result is a series in constant 2015 euros.

A note on the choice of deflator: HICP is the *consumer* price index. A producer or energy-specific deflator would yield slightly different values. HICP is appropriate here because the analysis is interested in price movements relative to general purchasing power, not relative to producer costs.

The effect of deflation on the price series is shown in Section 4.

In [195]:
hicp_base = hicp.loc[hicp['TIME_PERIOD'].dt.year == 2015, "OBS_VALUE"].mean()
hicp_base

np.float64(99.99833333333333)

In [196]:
electricity_price["year_month"] = electricity_price["Date"].dt.to_period("M")
hicp['TIME_PERIOD'] = pd.PeriodIndex(hicp['TIME_PERIOD'], freq='M')
prices = electricity_price.merge(
    hicp[['TIME_PERIOD', 'OBS_VALUE']],   
    left_on='year_month',
    right_on='TIME_PERIOD',
    how='left'
)
prices

,Country,ISO3 Code,Date,Price (EUR/MWhe),year_month,TIME_PERIOD,OBS_VALUE
0,Bulgaria,BGR,2016-10-01,35.53,2016-10,2016-10,98.46
1,Bulgaria,BGR,2016-10-02,35.53,2016-10,2016-10,98.46
2,Bulgaria,BGR,2016-10-03,35.53,2016-10,2016-10,98.46
3,Bulgaria,BGR,2016-10-04,35.53,2016-10,2016-10,98.46
4,Bulgaria,BGR,2016-10-05,35.53,2016-10,2016-10,98.46
...,...,...,...,...,...,...,...
3374,Bulgaria,BGR,2025-12-27,97.92,2025-12,2025-12,143.83
3375,Bulgaria,BGR,2025-12-28,93.65,2025-12,2025-12,143.83
3376,Bulgaria,BGR,2025-12-29,107.41,2025-12,2025-12,143.83
3377,Bulgaria,BGR,2025-12-30,97.62,2025-12,2025-12,143.83


In [197]:
prices["Real_Price"] = prices["Price (EUR/MWhe)"] * (hicp_base / prices["OBS_VALUE"]) 
prices

,Country,ISO3 Code,Date,Price (EUR/MWhe),year_month,TIME_PERIOD,OBS_VALUE,Real_Price
0,Bulgaria,BGR,2016-10-01,35.53,2016-10,2016-10,98.46,36.085119
1,Bulgaria,BGR,2016-10-02,35.53,2016-10,2016-10,98.46,36.085119
2,Bulgaria,BGR,2016-10-03,35.53,2016-10,2016-10,98.46,36.085119
3,Bulgaria,BGR,2016-10-04,35.53,2016-10,2016-10,98.46,36.085119
4,Bulgaria,BGR,2016-10-05,35.53,2016-10,2016-10,98.46,36.085119
...,...,...,...,...,...,...,...,...
3374,Bulgaria,BGR,2025-12-27,97.92,2025-12,2025-12,143.83,68.079238
3375,Bulgaria,BGR,2025-12-28,93.65,2025-12,2025-12,143.83,65.110505
3376,Bulgaria,BGR,2025-12-29,107.41,2025-12,2025-12,143.83,74.677195
3377,Bulgaria,BGR,2025-12-30,97.62,2025-12,2025-12,143.83,67.870662


In [198]:
prices['OBS_VALUE'].isna().sum()
prices['OBS_VALUE'].describe()

count    3379.000000
mean      117.344578
std        15.953651
min        98.360000
25%       103.730000
50%       108.200000
75%       135.440000
max       143.830000
Name: OBS_VALUE, dtype: float64

In [199]:
prices['Real_Price'].describe()

count    3379.000000
mean       75.617328
std        62.122993
min         2.429990
25%        37.882119
50%        54.763359
75%        86.739024
max       555.046217
Name: Real_Price, dtype: float64

In [200]:
prices.groupby(prices['Date'].dt.year).agg({
    'Price (EUR/MWhe)': 'mean',
    'Real_Price': 'mean'
})

,Price (EUR/MWhe),Real_Price
Date,,
2016,37.856087,38.366675
2017,39.668274,39.722420
2018,39.893233,38.874370
2019,47.474986,45.212686
2020,39.292814,36.958822
2021,108.701945,98.444288
2022,252.825315,203.839600
2023,103.843616,77.506718
2024,102.569727,74.398540


### 3.6 Aligning date ranges, merging, and post-merge missingness

At this stage the analysis has two daily series ready to combine: the price frame (now containing nominal price, HICP, and real price after Section 3.5) and the national weather frame (one column per variable, after the weighted aggregation in Section 3.4). Both are indexed by date, and both have already been filtered to the October 2016 – December 2025 window at load time. The next step is to merge them on date into a single analytical dataframe.

The merge is performed using a left join on the price frame, with the weather frame attached on the right. The choice of left join over inner join is deliberate: the price series is the variable being studied, and the analytical dataset must be defined by which days have prices, not by which days have both prices and weather. If a day has price data but no matching weather row, that gap surfaces as a NaN in the merged frame and can be diagnosed; an inner join would silently drop the row and obscure the issue.

After merging, the second missingness check is performed — the post-merge moment described at the start of this section. The per-source check in Section 3.2 told us in advance which source could plausibly contribute NaNs to the merged frame; the post-merge check verifies that the actual count matches that expectation, which is a guard against silent errors in the merge itself.

The expected outcome:

- Some NaN price rows from the per-source check in Section 3.2 — these are the ENTSO-E gaps documented earlier, and they are dropped here. Imputation is rejected for the same reason it was rejected at the per-source stage: the price is the dependent variable of the analysis, and substituting fictitious values into it is not defensible.
- No NaNs in the weather columns. ERA5 reanalysis has full coverage, and the population-weighted index is well-defined for every day where all five city series are present.
- No NaNs in the HICP column. The Bulgarian HICP is complete across the analysis window, and every daily price has a matching monthly HICP value (verified in Section 3.5).

Any deviation from this expectation indicates a merge bug — for example, a date type mismatch between the two frames, or a weather frame that doesn't actually cover the full window. The check exists to catch exactly that kind of error before it propagates into the analysis.

The number of dropped rows is reported in the notebook output. The starting price series has 3,379 observations after window filtering; the final merged dataset has whatever remains after the price-side gaps are removed.

In [201]:
prices = prices.merge(weather_national , on="Date" , how='left' )
prices

,Country,ISO3 Code,Date,Price (EUR/MWhe),year_month,TIME_PERIOD,OBS_VALUE,Real_Price,temperature,precipitation,wind_speed,cloud_cover,snowfall
0,Bulgaria,BGR,2016-10-01,35.53,2016-10,2016-10,98.46,36.085119,18.471514,0.000000,4.070366,28.647443,0.0
1,Bulgaria,BGR,2016-10-02,35.53,2016-10,2016-10,98.46,36.085119,18.546951,0.000000,4.932986,16.822106,0.0
2,Bulgaria,BGR,2016-10-03,35.53,2016-10,2016-10,98.46,36.085119,18.230524,0.234315,6.187056,39.980435,0.0
3,Bulgaria,BGR,2016-10-04,35.53,2016-10,2016-10,98.46,36.085119,15.528293,2.365291,8.580813,62.018893,0.0
4,Bulgaria,BGR,2016-10-05,35.53,2016-10,2016-10,98.46,36.085119,11.472123,0.625225,11.953045,48.023527,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3374,Bulgaria,BGR,2025-12-27,97.92,2025-12,2025-12,143.83,68.079238,1.515136,0.000000,16.067292,67.792444,0.0
3375,Bulgaria,BGR,2025-12-28,93.65,2025-12,2025-12,143.83,65.110505,2.160068,0.121775,19.473101,43.649332,0.0
3376,Bulgaria,BGR,2025-12-29,107.41,2025-12,2025-12,143.83,74.677195,1.605713,0.000000,16.154065,18.378453,0.0
3377,Bulgaria,BGR,2025-12-30,97.62,2025-12,2025-12,143.83,67.870662,1.068996,0.000000,13.083620,16.341491,0.0


In [209]:
prices.isnull().sum()

Country             0
ISO3 Code           0
Date                0
Price (EUR/MWhe)    0
year_month          0
TIME_PERIOD         0
OBS_VALUE           0
Real_Price          0
temperature         0
precipitation       0
wind_speed          0
cloud_cover         0
snowfall            0
dtype: int64

In [208]:
prices['Date'].is_monotonic_increasing


True

In [211]:
prices['Date'].diff().value_counts()

Date
1 days    3378
Name: count, dtype: int64

In [212]:
prices.describe()

,Date,Price (EUR/MWhe),OBS_VALUE,Real_Price,temperature,precipitation,wind_speed,cloud_cover,snowfall
count,3379,3379.000000,3379.000000,3379.000000,3379.000000,3379.000000,3379.000000,3379.000000,3379.000000
mean,2021-05-17 00:00:00,91.950997,117.344578,75.617328,12.613035,1.779249,8.879332,49.041089,0.124433
min,2016-10-01 00:00:00,2.570000,98.360000,2.429990,-12.957876,0.000000,2.505275,0.000000,0.000000
25%,2019-01-23 12:00:00,39.790000,103.730000,37.882119,5.757468,0.000000,6.727708,22.435708,0.000000
50%,2021-05-17 00:00:00,62.920000,108.200000,54.763359,12.365395,0.168589,8.395469,48.131645,0.000000
75%,2023-09-08 12:00:00,114.580000,135.440000,86.739024,20.076880,1.862207,10.443808,74.569294,0.000000
max,2025-12-31 00:00:00,700.480000,143.830000,555.046217,30.034033,36.309095,25.655680,100.000000,10.835396
std,NaN,79.515853,15.953651,62.122993,8.340673,3.580174,2.981188,29.971645,0.674262


### 3.7 Feature engineering

Two derived columns are added to the merged dataset to support the stratified analyses in Section 8.

- **Season** is derived from the date column, using the meteorological convention: December–February is winter, March–May is spring, June–August is summer, September–November is autumn. This convention is preferred over the astronomical one because Bulgarian electricity demand follows calendar months, not celestial events: a heating system runs on December dates regardless of whether the solstice has passed.
- **Market period** is derived from the date column, using a four-period split: October 2016 – 2019 (pre-crisis baseline), 2020 (COVID shock), 2021–2022 (energy crisis), 2023–2025 (post-crisis normalisation). The boundaries of these periods are choices, not natural givens; the rationale for each boundary is laid out in Section 8.2.

The pre-crisis baseline period begins in October 2016 — the IBEX launch date — rather than at the start of a calendar year. This is consistent with the choice in Section 3.1 to use the actual data window rather than a rounded one. For analyses that compare full years (e.g. seasonal patterns), the partial 2016 quarter can be reported alongside but excluded from per-year averaging where this would produce a misleading comparison; this is handled at the point of use rather than baked into the dataset.

In [243]:
prices.loc[(prices["Date"].dt.month == 12) | (prices["Date"].dt.month <= 2), "Season"] = "Winter"
prices.loc[(prices["Date"].dt.month >= 3 ) & (prices["Date"].dt.month <= 5), "Season"] = "Spring"
prices.loc[(prices["Date"].dt.month >= 6 ) & (prices["Date"].dt.month <= 8), "Season"] = "Summer"
prices.loc[(prices["Date"].dt.month >= 9 ) & (prices["Date"].dt.month <= 11), "Season"] = "Autumn"
prices

,Country,ISO3 Code,Date,Price (EUR/MWhe),year_month,TIME_PERIOD,OBS_VALUE,Real_Price,temperature,precipitation,wind_speed,cloud_cover,snowfall,Season,Market_Period
0,Bulgaria,BGR,2016-10-01,35.53,2016-10,2016-10,98.46,36.085119,18.471514,0.000000,4.070366,28.647443,0.0,Autumn,Pre-crisis
1,Bulgaria,BGR,2016-10-02,35.53,2016-10,2016-10,98.46,36.085119,18.546951,0.000000,4.932986,16.822106,0.0,Autumn,Pre-crisis
2,Bulgaria,BGR,2016-10-03,35.53,2016-10,2016-10,98.46,36.085119,18.230524,0.234315,6.187056,39.980435,0.0,Autumn,Pre-crisis
3,Bulgaria,BGR,2016-10-04,35.53,2016-10,2016-10,98.46,36.085119,15.528293,2.365291,8.580813,62.018893,0.0,Autumn,Pre-crisis
4,Bulgaria,BGR,2016-10-05,35.53,2016-10,2016-10,98.46,36.085119,11.472123,0.625225,11.953045,48.023527,0.0,Autumn,Pre-crisis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3374,Bulgaria,BGR,2025-12-27,97.92,2025-12,2025-12,143.83,68.079238,1.515136,0.000000,16.067292,67.792444,0.0,Winter,Post-crisis
3375,Bulgaria,BGR,2025-12-28,93.65,2025-12,2025-12,143.83,65.110505,2.160068,0.121775,19.473101,43.649332,0.0,Winter,Post-crisis
3376,Bulgaria,BGR,2025-12-29,107.41,2025-12,2025-12,143.83,74.677195,1.605713,0.000000,16.154065,18.378453,0.0,Winter,Post-crisis
3377,Bulgaria,BGR,2025-12-30,97.62,2025-12,2025-12,143.83,67.870662,1.068996,0.000000,13.083620,16.341491,0.0,Winter,Post-crisis


In [244]:
prices.loc[(prices["Date"].dt.year < 2020), "Market_Period"] = "Pre-crisis"
prices.loc[(prices["Date"].dt.year >=  2020) & (prices["Date"].dt.year <= 2022), "Market_Period"] = "Crisis"
prices.loc[(prices["Date"].dt.year >= 2023) & (prices["Date"].dt.year <= 2025), "Market_Period"] = "Post-crisis"

In [245]:
prices

,Country,ISO3 Code,Date,Price (EUR/MWhe),year_month,TIME_PERIOD,OBS_VALUE,Real_Price,temperature,precipitation,wind_speed,cloud_cover,snowfall,Season,Market_Period
0,Bulgaria,BGR,2016-10-01,35.53,2016-10,2016-10,98.46,36.085119,18.471514,0.000000,4.070366,28.647443,0.0,Autumn,Pre-crisis
1,Bulgaria,BGR,2016-10-02,35.53,2016-10,2016-10,98.46,36.085119,18.546951,0.000000,4.932986,16.822106,0.0,Autumn,Pre-crisis
2,Bulgaria,BGR,2016-10-03,35.53,2016-10,2016-10,98.46,36.085119,18.230524,0.234315,6.187056,39.980435,0.0,Autumn,Pre-crisis
3,Bulgaria,BGR,2016-10-04,35.53,2016-10,2016-10,98.46,36.085119,15.528293,2.365291,8.580813,62.018893,0.0,Autumn,Pre-crisis
4,Bulgaria,BGR,2016-10-05,35.53,2016-10,2016-10,98.46,36.085119,11.472123,0.625225,11.953045,48.023527,0.0,Autumn,Pre-crisis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3374,Bulgaria,BGR,2025-12-27,97.92,2025-12,2025-12,143.83,68.079238,1.515136,0.000000,16.067292,67.792444,0.0,Winter,Post-crisis
3375,Bulgaria,BGR,2025-12-28,93.65,2025-12,2025-12,143.83,65.110505,2.160068,0.121775,19.473101,43.649332,0.0,Winter,Post-crisis
3376,Bulgaria,BGR,2025-12-29,107.41,2025-12,2025-12,143.83,74.677195,1.605713,0.000000,16.154065,18.378453,0.0,Winter,Post-crisis
3377,Bulgaria,BGR,2025-12-30,97.62,2025-12,2025-12,143.83,67.870662,1.068996,0.000000,13.083620,16.341491,0.0,Winter,Post-crisis
